# 3W Challenge Baseline: Multiclass Sample Classification

Build a sample-level classifier that predicts the operational label **0 through 9** for each labeled observation.

**Audience:** Python users who know basic pandas and classification concepts.

**Learning goals:** load fixed instance splits with 3W Toolkit, apply the
challenge label rules, train a default model five times, and summarize the
validation results. No hyperparameter search is performed.

## Prerequisites and outline

Prerequisites:

- install the repository with `pip install -e .`;
- copy `.env.example` to `.env` and configure the repository and complete
  3W Dataset v2.0.0 paths.

Outline:

1. Resolve the repository, dataset, and fixed split lists.
2. Remove unlabeled rows and map transient labels to steady fault labels.
3. Fit 3W Toolkit signal cleaning independently on each training fold.
4. Fit Toolkit mean imputation independently on each training fold.
5. Fit a fresh default decision tree on each of the five fixed splits.
6. Review metrics and the aggregated confusion matrix.

In [ ]:
import logging
import os
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from ThreeWToolkit.core.base_dataset import BaseDataset
from ThreeWToolkit.core.dataset_outputs import DatasetOutputs
from ThreeWToolkit.dataset import (
    ParquetDataset,
    ParquetDatasetConfig,
    SubsetDataset,
    TransformConfig,
    TransformedDataset,
)
from ThreeWToolkit.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from ThreeWToolkit.models import SklearnModelsConfig
from ThreeWToolkit.preprocessing import (
    CleanSignalsConfig,
    ImputeMissingConfig,
    SequentialPreprocessingAdapterConfig,
)
from ThreeWToolkit.trainer import SklearnTrainerConfig
from ThreeWToolkit.utils.data_utils import get_config_dataset_ini
from tqdm.auto import tqdm

# Jupyter renders logging's stderr stream in red, so hide routine Toolkit logs.
logging.getLogger("ThreeWToolkit").setLevel(logging.WARNING)

load_dotenv()

REPOSITORY_PATH = Path(os.environ["THREE_W_REPOSITORY_PATH"]).expanduser().resolve()
DATASET_PATH = Path(os.environ["THREE_W_DATASET_PATH"]).expanduser().resolve()
CHALLENGE_PATH = Path(os.environ["THREE_W_CHALLENGE_PATH"]).expanduser().resolve()
SPLIT_DIR = Path(os.environ["THREE_W_CHALLENGE_SPLIT_DIR"]).expanduser().resolve()
RANDOM_SEED = int(os.environ["THREE_W_CHALLENGE_RANDOM_SEED"])

print(f"Repository: {REPOSITORY_PATH}")
print(f"Dataset:   {DATASET_PATH}")
print(f"Challenge: {CHALLENGE_PATH}")
print(f"Splits:    {SPLIT_DIR}")

## 1. Load every instance from the fixed splits

Each text file contains the validation instances for one run. The other four
lists form training. Their union contains all 2,228 dataset instances. Because assignment is at the instance level, samples from one Parquet file never appear in both training and validation.

In [ ]:
def read_validation_lists(split_dir: Path) -> dict[int, set[str]]:
    """Read the five one-path-per-line toolkit file lists."""
    validation_lists = {}
    for split_number in range(1, 6):
        path = split_dir / f"validation_split_{split_number}.txt"
        instances = {
            line.strip()
            for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        }
        validation_lists[split_number] = instances

    all_entries = [
        instance for instances in validation_lists.values() for instance in instances
    ]
    if len(all_entries) != len(set(all_entries)):
        raise ValueError("An instance occurs in more than one validation list.")
    return validation_lists


validation_lists = read_validation_lists(SPLIT_DIR)
instance_files = sorted(set().union(*validation_lists.values()))
if len(instance_files) != 2_228:
    raise ValueError(f"Expected all 2,228 instances, found {len(instance_files):,}.")

print(f"Using {len(instance_files):,} instance files.")
pd.Series(
    {split: len(files) for split, files in validation_lists.items()},
    name="validation_instances",
).to_frame()

## 2. Apply the challenge cleaning and label rules

`ParquetDataset` exposes the original labels, so the challenge rules are
applied before any fitted preprocessing. The loader:

1. drops missing labels;
2. maps transient labels with the toolkit's configured offset (`105 → 5`);
3. keeps mapped labels `0` through `9`.

All eligible observations are loaded without sampling. Signal-cleaning thresholds and missing-value means are deliberately not computed here. They are fitted independently inside each validation run from that run's training instances only, so validation instances cannot influence preprocessing.

In [ ]:
DATASET_CONFIG = get_config_dataset_ini()
FEATURE_COLUMNS = [
    column
    for column in DATASET_CONFIG["COLUMNS_DATA_FILES"]
    if column not in {"timestamp", "class", "state"}
]
TRANSIENT_OFFSET = int(DATASET_CONFIG["TRANSIENT_OFFSET"])


class ChallengeDataset(BaseDataset):
    """Small in-memory adapter for Toolkit preprocessors and trainers."""

    def __init__(self, events: list[DatasetOutputs]):
        self.events = events

    def __len__(self) -> int:
        return len(self.events)

    def __getitem__(self, index: int) -> DatasetOutputs:
        return self.events[index]


def load_challenge_samples(
    dataset_path: Path,
    files: list[str],
    *,
    binary: bool,
) -> tuple[ChallengeDataset, int]:
    """Load and relabel every eligible observation without sampling."""
    dataset = ParquetDataset(
        ParquetDatasetConfig(
            path=dataset_path,
            split="list",
            file_list=files,
            columns=FEATURE_COLUMNS,
            target_column="class",
        )
    )

    events = []
    unlabeled_removed = 0
    for item in tqdm(dataset, total=len(dataset), unit="instance"):
        raw_label = item.label
        if raw_label is None:
            raise ValueError("The configured target column did not produce labels.")
        valid_positions = np.flatnonzero(raw_label.notna().to_numpy())
        unlabeled_removed += len(raw_label) - len(valid_positions)
        if not len(valid_positions):
            continue

        signal = item.signal.iloc[valid_positions].copy()
        labels = (
            raw_label.iloc[valid_positions].astype("int64").to_numpy()
            % TRANSIENT_OFFSET
        )

        allowed = np.isin(labels, np.arange(10))
        signal = signal.iloc[np.flatnonzero(allowed)].copy()
        labels = labels[allowed]
        if not len(labels):
            continue

        target = labels
        if binary:
            target = (target != 0).astype("int8")
        file_name = Path(item.metadata["file_name"]).as_posix()
        events.append(
            DatasetOutputs(
                signal=signal.reset_index(drop=True),
                label=pd.Series(target, name="target"),
                metadata={**item.metadata, "file_name": file_name},
            )
        )

    if not events:
        raise RuntimeError("No labeled samples were loaded.")
    return ChallengeDataset(events), unlabeled_removed

In [ ]:
samples, unlabeled_removed = load_challenge_samples(
    DATASET_PATH,
    instance_files,
    binary=False,
)
retained_observations = sum(len(event.signal) for event in samples)
print(f"Removed {unlabeled_removed:,} unlabeled observations before preprocessing.")
print(
    f"Retained {retained_observations:,} observations from {len(samples):,} instances."
)
pd.concat(
    [event.label for event in samples if event.label is not None],
    ignore_index=True,
).value_counts().sort_index().rename("samples").to_frame()

## 3. Fit and evaluate all five splits

The model uses the 3W Toolkit's `SklearnTrainer` with scikit-learn's
`DecisionTreeClassifier`. Its model hyperparameters stay at their defaults; only
`random_state` is fixed for reproducibility. A fresh preprocessing pipeline is fitted for every split. Toolkit `CleanSignals` thresholds and mean-imputation values are learned from the training fold only and then applied unchanged to validation. The "global average" is therefore global only within the current training fold, never across the complete dataset. Decision trees do not need feature scaling, so no normalization is used.

`TRAINING_SAMPLE_STRIDE` controls training size without changing validation. Use `1` for every training observation or `N` for every Nth training observation. For example, `10` keeps approximately one tenth of each split's training rows. Macro-F1 and balanced accuracy give every class equal weight, making them more informative than raw accuracy for the imbalanced 3W labels.

In [ ]:
# 1 uses every training row; N uses every Nth training row.
TRAINING_SAMPLE_STRIDE = 1
if not isinstance(TRAINING_SAMPLE_STRIDE, int) or TRAINING_SAMPLE_STRIDE < 1:
    raise ValueError(
        "TRAINING_SAMPLE_STRIDE must be an integer greater than or equal to 1."
    )

print(f"Training sample stride: {TRAINING_SAMPLE_STRIDE}")


def fresh_fold_preprocessor():
    """Build a Toolkit pipeline whose statistics are fitted per training fold."""
    return TransformConfig(
        pre_processing=SequentialPreprocessingAdapterConfig(
            steps=[
                CleanSignalsConfig(),
                ImputeMissingConfig(strategy="mean"),
            ]
        )
    ).build()


def fresh_default_trainer():
    """Build a fresh Toolkit trainer around a default decision tree."""
    return SklearnTrainerConfig(
        config_model=SklearnModelsConfig(
            model_type=DecisionTreeClassifier,
            model_params={"random_state": RANDOM_SEED},
        ),
        seed=RANDOM_SEED,
    ).build()


def stride_training_rows(dataset: BaseDataset, stride: int) -> BaseDataset:
    """Keep every Nth observation within each training instance."""
    if stride == 1:
        return dataset

    def select_rows(event: DatasetOutputs) -> DatasetOutputs:
        positions = np.arange(0, len(event.signal), stride)
        label = None
        if event.label is not None:
            label = event.label.iloc[positions].reset_index(drop=True)
        return DatasetOutputs(
            signal=event.signal.iloc[positions].reset_index(drop=True),
            label=label,
            metadata=event.metadata,
        )

    return TransformedDataset(dataset, select_rows)


def evaluate_five_splits(
    samples: ChallengeDataset,
    split_files: dict[int, set[str]],
    labels: list[int],
    training_sample_stride: int,
) -> tuple[pd.DataFrame, np.ndarray, dict]:
    """Fit fold-only preprocessing and a fresh model on every fixed split."""
    instance_names = [str(event.metadata["file_name"]) for event in samples]
    rows = []
    aggregate_confusion = np.zeros((len(labels), len(labels)), dtype=int)
    effective_params = None

    for split_number in range(1, 6):
        validation_instances = split_files[split_number]
        validation_indices = [
            index
            for index, name in enumerate(instance_names)
            if name in validation_instances
        ]
        train_indices = [
            index
            for index, name in enumerate(instance_names)
            if name not in validation_instances
        ]
        raw_train_dataset = SubsetDataset(samples, train_indices)
        raw_validation_dataset = SubsetDataset(samples, validation_indices)

        available_train_samples = sum(
            len(samples[index].signal) for index in train_indices
        )
        train_samples = sum(
            (len(samples[index].signal) + training_sample_stride - 1)
            // training_sample_stride
            for index in train_indices
        )
        validation_samples = sum(
            len(samples[index].signal) for index in validation_indices
        )

        split_started = perf_counter()
        print(f"\nSplit {split_number}/5", flush=True)
        print(
            f"  Training rows: {train_samples:,} of {available_train_samples:,} available "
            f"(stride={training_sample_stride})",
            flush=True,
        )
        print(f"  Validation rows: {validation_samples:,}", flush=True)

        print(
            "  Fitting Toolkit cleaning and mean imputation on training only...",
            flush=True,
        )
        preprocessor = fresh_fold_preprocessor()
        preprocessor.fit(raw_train_dataset)
        train_dataset = stride_training_rows(
            preprocessor.transform(raw_train_dataset), training_sample_stride
        )
        validation_dataset = preprocessor.transform(raw_validation_dataset)

        trainer = fresh_default_trainer()
        effective_params = dict(trainer.model.get_params())
        print("  Fitting the decision tree...", flush=True)
        trainer.train(train_dataset)
        print("  Predicting the validation fold...", flush=True)
        prediction_result = trainer.predict(validation_dataset)
        if prediction_result.y_true is None:
            raise RuntimeError("Validation labels are required for evaluation.")
        y_validation = prediction_result.y_true
        prediction = prediction_result.y_pred

        row = {
            "split": split_number,
            "available_train_samples": available_train_samples,
            "train_samples": train_samples,
            "validation_samples": validation_samples,
            "accuracy": accuracy_score(y_validation, prediction),
            "balanced_accuracy": balanced_accuracy_score(y_validation, prediction),
            "macro_f1": f1_score(
                y_validation,
                prediction,
                labels=labels,
                average="macro",
                zero_division=0,
            ),
        }
        rows.append(row)
        aggregate_confusion += confusion_matrix(y_validation, prediction, labels=labels)
        print(
            f"  Complete in {perf_counter() - split_started:,.1f}s: "
            f"balanced_accuracy={row['balanced_accuracy']:.4f}, "
            f"macro_f1={row['macro_f1']:.4f}",
            flush=True,
        )

    return pd.DataFrame(rows), aggregate_confusion, effective_params

In [ ]:
fold_results, aggregate_confusion, effective_params = evaluate_five_splits(
    samples,
    validation_lists,
    labels=list(range(10)),
    training_sample_stride=TRAINING_SAMPLE_STRIDE,
)
display(fold_results.round(4))
display(fold_results.mean(numeric_only=True).rename("five_split_mean").to_frame())

print("DecisionTree parameters used in every split (no search):")
effective_params

## 4. Aggregate the confusion matrices

Summing counts before row normalization gives every validation observation one
vote across the five non-overlapping validation folds.

In [ ]:
row_totals = aggregate_confusion.sum(axis=1, keepdims=True)
normalized_confusion = np.divide(
    aggregate_confusion,
    row_totals,
    out=np.zeros_like(aggregate_confusion, dtype=float),
    where=row_totals != 0,
)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    normalized_confusion,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=list(range(10)),
    yticklabels=list(range(10)),
    ax=ax,
)
ax.set(
    xlabel="Predicted class", ylabel="True class", title="Five-split confusion matrix"
)
plt.show()

## Exercise: find the hardest classes

Recall is the diagonal of the row-normalized confusion matrix. Rank the classes
to identify which event labels the default model recognizes least reliably.

In [ ]:
# Answer scaffold
recall_by_class = pd.Series(
    np.diag(normalized_confusion),
    index=list(range(10)),
    name="recall",
)
recall_by_class.sort_values().to_frame()

## Pitfalls and extensions

- **Do not fill missing labels before filtering.** The challenge rule is to
  remove them; the toolkit's default label processing uses forward/backward
  filling, so this tutorial handles labels explicitly.
- **Do not split rows randomly.** Neighboring samples are highly correlated.
  These folds keep whole instances together and also group real files from the
  same well and day.
- This default tree is a teaching baseline, not a production model.

Optional extension: keep the split files fixed and compare a window-based
feature extractor or a stricter whole-well validation design. Treat that as a
new experiment rather than tuning on these validation results.